# Metadata Ingestion - Brazilian Ecommerce

This notebook registers the Brazilian Ecommerce (Olist) dataset source into the metadata control tables (`statedb.databaselist` and `statedb.objectlist`) and inserts all archive file entries for pipeline ingestion.

In [0]:
# Cell 1: Register Brazilian Ecommerce source in databaselist
# Follow the same pattern as the existing AdventureWorks entry (DatabaseID=1)

from pyspark.sql.types import *
from datetime import datetime

# Check if a databaselist entry for ecommerce already exists
existing = spark.sql("""
    SELECT DatabaseID FROM mde_dev.statedb.databaselist 
    WHERE DatabaseName = 'ecommerce' OR ShortName = 'BEC'
""").collect()

if existing:
    db_id = existing[0][0]
    print(f"Brazilian Ecommerce source already exists in databaselist with DatabaseID={db_id}. Skipping insert.")
else:
    # Insert new databaselist entry for Brazilian Ecommerce
    spark.sql("""
        INSERT INTO mde_dev.statedb.databaselist (
            DatabaseID, ServerName, DatabaseName, DatabaseUser, ShortName, SourceType,
            DataFactoryName, IRName, MainDataContact, MainITContact, Notes, SecretName,
            ExtractType, DataLakeRootPath, DataLakeEndpoint, DataLakeContainerName, DataLakeCatalogName,
            CreatedDate, ModifiedDate, LastUpdateDate, LastUpdatedBy
        ) VALUES (
            2,
            'Unity Catalog Volume',
            'ecommerce',
            NULL,
            'BEC',
            'UNITY_CATALOG_VOLUME',
            NULL,
            NULL,
            'vamshialwayskush@gmail.com',
            'vamshialwayskush@gmail.com',
            'Brazilian Ecommerce (Olist) dataset stored in Unity Catalog Volume',
            NULL,
            'Full',
            '/Volumes/mde_dev/sources/ecommerce/archive/',
            NULL,
            'sources',
            'mde_dev',
            current_timestamp(),
            current_timestamp(),
            current_timestamp(),
            'vamshialwayskush@gmail.com'
        )
    """)
    db_id = 2
    print(f"Inserted Brazilian Ecommerce source into databaselist with DatabaseID={db_id}")

# Display the updated databaselist
display(spark.sql("SELECT * FROM mde_dev.statedb.databaselist ORDER BY DatabaseID"))

In [0]:
# Cell 2: Register all ecommerce archive files in objectlist
# Following the same pattern as existing AdventureWorks entries (ObjectID 1-16)

# Get the next available ObjectID
max_id_row = spark.sql("SELECT COALESCE(MAX(ObjectID), 0) AS max_id FROM mde_dev.statedb.objectlist").collect()
next_id = max_id_row[0]["max_id"] + 1
print(f"Next available ObjectID: {next_id}")

# Define all files in the ecommerce archive folder
archive_base_path = "/Volumes/mde_dev/sources/ecommerce/archive/"
ecommerce_files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]

# Build INSERT statements for each file
# Pattern follows existing AdventureWorks entries:
#   ObjectSchema = "sources" (the volume schema)
#   ObjectName = filename without .csv extension
#   StageTableName = lowercased ObjectName
#   LoadGroup = "BEC" (matching ShortName from databaselist)
#   ExtractFileType = "CSV", FileHeader = true, delimiter = comma

inserted_ids = []
for filename in ecommerce_files:
    object_name = filename.replace(".csv", "")
    file_path = f"{archive_base_path}{filename}"
    
    # Check if this file is already registered
    existing = spark.sql(f"""
        SELECT ObjectID FROM mde_dev.statedb.objectlist 
        WHERE FilePath = '{file_path}'
    """).collect()
    
    if existing:
        print(f"  Skipping {filename} - already registered as ObjectID={existing[0][0]}")
        continue
    
    object_id = next_id
    next_id += 1
    
    spark.sql(f"""
        INSERT INTO mde_dev.statedb.objectlist (
            ObjectID, DatabaseId, ObjectSchema, ObjectName, IncludeInLoad, LoadIncremental,
            HardDeleteCapture, WhereQueryPart, WaterMarkColumn, WaterMarkColumnDataType,
            WaterMarkType, WaterMarkQueriedFromSource, WaterMarkOverlapValue, ClientValidated,
            LoadGroup, StageSchemaName, StageTableName, StageDistribution, StageTableType,
            IsStageTableType2, ExtractFileType, FileHeader, FilePath, FileColDelimiter,
            FileQuoteCharacter, FileEscapeCharacter, IncrReloadNextRun,
            CreatedDate, ModifiedDate, LastModifiedBy
        ) VALUES (
            {object_id},
            2,
            'sources',
            '{object_name}',
            true,
            false,
            false,
            NULL,
            NULL,
            NULL,
            NULL,
            NULL,
            NULL,
            true,
            'BEC',
            'bronze',
            '{object_name.lower()}',
            NULL,
            'Delta',
            false,
            'CSV',
            true,
            '{file_path}',
            ',',
            '"',
            '\\\\',
            false,
            current_timestamp(),
            current_timestamp(),
            'vamshialwayskush@gmail.com'
        )
    """)
    inserted_ids.append(object_id)
    print(f"  Inserted ObjectID={object_id}: {object_name} -> {file_path}")

print(f"\nTotal new entries inserted: {len(inserted_ids)}")
if inserted_ids:
    print(f"ObjectIDs: {inserted_ids}")

In [0]:
# Cell 3: Verify the inserted entries

print("=== databaselist entries ===")
display(spark.sql("SELECT DatabaseID, ShortName, DatabaseName, SourceType, DataLakeRootPath FROM mde_dev.statedb.databaselist ORDER BY DatabaseID"))

print("\n=== New objectlist entries for Brazilian Ecommerce (DatabaseId=2) ===")
display(spark.sql("""
    SELECT ObjectID, DatabaseId, ObjectSchema, ObjectName, IncludeInLoad, LoadGroup,
           StageSchemaName, StageTableName, StageTableType, ExtractFileType, 
           FileHeader, FilePath, FileColDelimiter
    FROM mde_dev.statedb.objectlist 
    WHERE DatabaseId = 2
    ORDER BY ObjectID
"""))

print("\n=== Total objectlist entries ===")
total = spark.sql("SELECT COUNT(*) as total FROM mde_dev.statedb.objectlist").collect()
print(f"Total objectlist entries: {total[0]['total']}")